In [1]:
# LLM based Research and Report Generation

# GOAL===>
# Topics covered in creating this project are:
# 1. Human in the loop
# 2. Map Reduce
# 3. External Tool Calls

In [58]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_core.tools import tool

In [59]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(max_results= 3)

In [60]:
# defining the state of my multi agentic system

import operator
from typing import TypedDict, Annotated, List


class ResearchState(TypedDict):
    question: str # the original question

    sub_questions: List[str] # the list of all the sub questions

    # the researcher node writes down its research results parallely.
    research_results: Annotated[List[str], operator.add] 

    # Critic writes a score (1-10)
    quality_score: int # if the score is less than a threshold value, the result will be rewritten.

    # to prevent infinite critic-> rewrite loop
    retries: int

    # Final Research Report
    final_report: str

    # Human approves the final report or asks for rewriting.
    human_feedback: str

In [61]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model = "gemini-flash-latest", temperature = 0.7, max_retries = 2)

In [62]:
def planner(state: ResearchState):
    question = state["question"]
    print("Planning:")
    # We're using a plain string prompt here, not a chat template.
    # For simple single-turn calls like this, it's cleaner and sufficient.
    prompt = f"""You are a research planner. A user has asked the following question:

"{question}"

Break this question into 3 specific questions which if queried alone will still be able to give some answer on the question.
Each sub-question should cover a distinct angle.

Respond with ONLY the 3 sub-questions, one per line, no numbering, no extra text."""

    response = model.invoke(prompt)
    sub_questions = (response.content[0]["text"]).split("\n").strip()
    return {"sub_questions": sub_questions}

In [63]:
def researcher(state: ResearchState):
    print("Researching: ")

    sub_question = state["sub_question"]

    results = tavily.invoke(sub_question)

    merged_results = "\n\n".join([r["content"] for r in results])

    return {"research_results": [merged_results]}


from langgraph.types import Send
def map_research(state: ResearchState):
    return [
        Send("researcher", {"sub_question": q})
        for q in state["sub_questions"]
    ]


In [64]:
def critic(state: ResearchState):
    research = "\n\n".join(state["research_results"])

    prompt = f"""You are a research quality critic.

Here is the compiled research for the question: "{state['question']}"

{research}

Evaluate this research on these criteria:
- Does it cover the question from multiple angles?
- Is the information substantive and not superficial?
- Are there clear gaps or contradictions?

Respond with ONLY a single integer score from 1 to 10. Nothing else."""

    response = model.invoke(prompt)

    
    try:
        score = int(response.content[0]["text"].strip())
    except ValueError:
        # If parsing fails for any reason, default to 7 to avoid blocking the loop
        score = 7

In [65]:
def writer(state: ResearchState):
    research = "\n\n".join(state["research_results"])
    revision_count = state["revision_count"]
    
    previous = ""
    if revision_count > 0 and state.get("final_report"):
        previous = f"""
        This is a revision. Previous report for reference:
        {state['final_report']}

        Improve it based on the research provided.
        """
    prompt = f"""You are a research report writer.

    Question: "{state['question']}"

    Research findings:
    {research}

    {previous}

    Write a clear, structured report answering the question. 
    Use the research findings as your source.
    Format: short intro, 3 key sections, brief conclusion.
    Keep it under 400 words."""
        
    response = model.invoke(prompt)

    print(f">> Writer produced report (revision {revision_count})")

    return {
        "final_report": response.content[0]["text"].strip(),
        # increment revision count every time writer runs
        "revision_count": revision_count + 1
    }

def should_revise(state: ResearchState):
        
    if state["revision_count"] >= 2:
        print(">> Max revisions reached, exiting loop")
        return END

    # If score is 7 or above, report is good enough
    if state["quality_score"] >= 7:
        print(">> Quality acceptable, exiting loop")
        return END

    print(">> Quality too low, sending back to writer")
    return "writer"

In [66]:
def critic(state: ResearchState):
    research = "\n\n".join(state["research_results"])

    prompt = f"""You are a research quality critic.

Here is the compiled research for the question: "{state['question']}"

{research}

Evaluate this research on these criteria:
- Does it cover the question from multiple angles?
- Is the information substantive and not superficial?
- Are there clear gaps or contradictions?

Respond with ONLY a single integer score from 1 to 10. Nothing else."""

    response = model.invoke(prompt)

    try:
        score = int(response.content.strip())
    except ValueError:
        # If parsing fails for any reason, default to 7 to avoid blocking the loop
        score = 0
    
    return {"quality_score": score}

In [67]:
def writer(state: ResearchState):
    research = "\n\n".join(state["research_results"])
    revision_count = state["revision_count"]
    
    previous = ""
    if revision_count > 0 and state.get("final_report"):
        previous = f"""
        This is a revision. Previous report for reference:
        {state['final_report']}

        Improve it based on the research provided.
        """
    prompt = f"""You are a research report writer.

    Question: "{state['question']}"

    Research findings:
    {research}

    {previous}

    Write a clear, structured report answering the question. 
    Use the research findings as your source.
    Format: short intro, 3 key sections, brief conclusion.
    Keep it under 400 words."""
        
    response = model.invoke(prompt)

    print(f">> Writer produced report (revision {revision_count})")

    return {
        "final_report": response.content.strip(),
        # increment revision count every time writer runs
        "revision_count": revision_count + 1
    }

def should_revise(state: ResearchState):
        
    if state["revision_count"] >= 2:
        print(">> Max revisions reached, exiting loop")
        return END

    # If score is 7 or above, report is good enough
    if state["quality_score"] >= 7:
        print(">> Quality acceptable, exiting loop")
        return END

    print(">> Quality too low, sending back to writer")
    return "writer"

In [69]:
initial_state = {
    "question": "What is agentic AI and how does it work?",
    "sub_questions": [],
    "research_results": [],
    "quality_score": 0,
    "revision_count": 0,
    "final_report": "",
    "human_feedback": ""
}
builder = StateGraph(ResearchState)

builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("critic", critic)
builder.add_node("writer", writer)

builder.set_entry_point("planner")
builder.add_conditional_edges("planner", map_research)
builder.add_edge("researcher", "critic")

# critic no longer has a fixed edge to writer
# instead it routes dynamically via should_revise
builder.add_conditional_edges("critic", should_revise)

# writer always goes back to critic for re-evaluation
builder.add_edge("writer", "critic")

graph = builder.compile()
result = graph.invoke(initial_state)

print("\n========= FINAL REPORT =========")
print(result["final_report"])

Planning:


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}